In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np, pandas as pd, json, time
from pathlib import Path
from datetime import datetime
DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
MODELS = [f'{a}_{v}' for v in ['5class_cw', '5class_smote'] for a in ['rf', 'xgb', 'dnn']]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
TABLES = Path(REPO) / 'results' / 'tables'
PREFIX = 'paper_inputs'

def find_proba_file(dataset, model_name, split):
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / f'{model_name}_{split}_proba.npy'
        if p.exists():
            return p
    raise FileNotFoundError(f'{dataset}/{model_name}_{split}_proba.npy')

def macro_f1(y, pred):
    conf = np.bincount(y * 5 + pred, minlength=25).reshape(5, 5).astype(float)
    tp = np.diag(conf); fp = conf.sum(0) - tp; fn = conf.sum(1) - tp
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
    return float(f1.mean()), conf

rows, flips = [], []
for ds in DATASETS:
    y = np.load(f'{REPO}/data/processed/{ds}/y_test_5class.npy')
    eval_idx = np.load(Path(REPO) / 'shap_values' / ds / 'canonical_eval_idx.npy')
    for m in MODELS:
        P_raw = np.load(find_proba_file(ds, m, 'test')); P_cal = np.load(Path(REPO) / 'calibrators' / ds / f'{m}_test_proba_calibrated.npy')
        pr, pc = P_raw.argmax(1), P_cal.argmax(1)
        f_raw, conf_raw = macro_f1(y, pr); f_cal, conf_cal = macro_f1(y, pc)
        row = {'dataset': ds, 'model': m, 'macro_f1_raw_argmax': f_raw, 'macro_f1_calibrated_argmax': f_cal,
               'accuracy_raw_argmax': float((pr == y).mean()), 'accuracy_calibrated_argmax': float((pc == y).mean()),
               'pct_flipped_full_test': 100 * float((pr != pc).mean()),
               'pct_flipped_canonical': 100 * float((pr[eval_idx] != pc[eval_idx]).mean())}
        for c in range(5):
            m_c = y == c
            row[f'recall_{CLASS_NAMES_5[c]}_raw'] = float((pr[m_c] == c).mean()) if m_c.any() else float('nan')
            row[f'recall_{CLASS_NAMES_5[c]}_calibrated'] = float((pc[m_c] == c).mean()) if m_c.any() else float('nan')
            row[f'predicted_share_{CLASS_NAMES_5[c]}_raw'] = float((pr == c).mean())
            row[f'predicted_share_{CLASS_NAMES_5[c]}_calibrated'] = float((pc == c).mean())
        rows.append(row)
        # where do the flipped predictions go
        fl = pr != pc
        for a in range(5):
            for b in range(5):
                n = int(((pr == a) & (pc == b) & fl).sum())
                if n:
                    flips.append({'dataset': ds, 'model': m, 'from_raw': CLASS_NAMES_5[a], 'to_calibrated': CLASS_NAMES_5[b], 'n': n,
                                  'share_of_flips': n / max(int(fl.sum()), 1)})
        print(f'{ds:15s} {m:18s} F1 raw={f_raw:.4f} cal={f_cal:.4f}  acc raw={row["accuracy_raw_argmax"]:.4f} cal={row["accuracy_calibrated_argmax"]:.4f}  flipped={row["pct_flipped_full_test"]:5.1f}%  '
              + ' '.join(f'{CLASS_NAMES_5[c][:3]} {row[f"recall_{CLASS_NAMES_5[c]}_raw"]:.2f}->{row[f"recall_{CLASS_NAMES_5[c]}_calibrated"]:.2f}' for c in range(5)))
df = pd.DataFrame(rows); df.to_csv(TABLES / f'{PREFIX}_decision_change.csv', index=False)
dfl = pd.DataFrame(flips); dfl.to_csv(TABLES / f'{PREFIX}_decision_flips.csv', index=False)
print('\nlargest flip routes per dataset:')
print(dfl.sort_values('n', ascending=False).groupby('dataset').head(3).to_string(index=False))


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"
import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '15_decision_change_under_calibration.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')
!git add notebooks/15_decision_change_under_calibration.ipynb results/tables/paper_inputs_decision_change.csv results/tables/paper_inputs_decision_flips.csv
!git status --short | head -20
!git commit -m "Notebook 15: decision change under per-class calibration. Per-class recall and macro-F1 under raw vs calibrated argmax on full test sets; flip routes between classes"
!git push origin main
!git log --oneline -2
